In [1]:
# -*- coding: utf-8 -*-
"""
════════════════════════════════════════════════════════════════════════════
П.3 «Итогов второй встречи с оппонентом»
Метод независимых компонент (ICA) на рядах PP и PR — с проверкой на всех
парах интервалов (RR, PP, PR), двух наборах отведений и по фазам пробы

ЕДИНЫЙ ФАЙЛ. Ничего импортировать из соседних файлов не нужно; скрытого
кэша нет — ряды каждый раз строятся заново из записей.

Установка:
    pip install numpy scipy scikit-learn pyedflib matplotlib

Запуск из командной строки:
    python П3_ICA_единый_код.py /путь/к/записям [--out results_p3]

Запуск в Jupyter / Google Colab:
    1) !pip install pyedflib   (остальное в Colab уже есть)
    2) выполнить ячейку с этим кодом целиком (расчёт при этом не начнётся)
    3) в следующей ячейке:  res = run_p3('/путь/к/записям')

Для записей с другого аппарата поправьте блок НАСТРОЙКИ ниже: номер канала
отведения II, таблицу отведений из П.7, пороги числа циклов, окна фаз.
Частота дискретизации берётся из самой записи.

Протокол пробы (пассивный ортостаз на поворотном столе): около 5 мин
записи лёжа, подъём стола до 45° без прерывания записи, около 5 мин в
положении подъёма. Метки «Этап»: около 05:00 — начало подъёма, около
05:10–05:20 — конец подъёма, около 10:00 — конец эксперимента. Покой — от
30 с до (начало подъёма − 5 с); наклон — от (конец подъёма + 15 с) до
(конец подъёма + 275 с). Отведения: каналы 0, 1, 2 = стандартные I, II, III.

Что считается
  1. Основной расчёт: ICA на паре [PP, PR],
     отведение II, запись целиком. Остаток ICA и остаток регрессии PR на PP
     сравниваются по корреляции с быстрой частью PR (PR минус скользящее
     среднее по 15 циклам).
  2. То же на лучшем отведении каждой записи — том, где зубцы P, Q, R, S, T
     определяются надёжнее всего (критерий чистоты разметки, п. 7).
  3. Все пары интервалов. Между рядами есть точное тождество
     RR(k) = PP(k) + PR(k+1) − PR(k), поэтому три ряда — не три независимых
     наблюдения, и ICA применяется к парам: PP+PR, RR+PR и контрольная пара
     PP+RR, где собственная составляющая RR (разность соседних PR) известна
     точно.
  4. Варианты: запись целиком; без выбросов; отдельно покой и наклон;
     покой и наклон после удаления медленной части рядов; порог 250 циклов.
  5. Негауссовость рядов, доля разброса от смены фазы, доля медленной
     части внутри фаз; доля циклов с найденным зубцом P в покое и при наклоне.
  6. Проверка на синтетике с известными источниками: при каких условиях
     ICA вообще может обогнать регрессию.
  7. Вариант без уточнения положений R и P (точность — один отсчёт, 2 мс):
     как погрешность измерения влияет на сравнение ICA и регрессии.
  8. Устойчивость вывода к случайному старту FastICA (20 стартов) и к
     настройкам алгоритма (нелинейности logcosh, exp, cube; параллельный и
     последовательный варианты).
  9. Охват: по каждой из 20 записей — сколько годных циклов в каждом
     варианте и почему запись не вошла в расчёт.
════════════════════════════════════════════════════════════════════════════
"""
!pip install numpy scipy scikit-learn pyedflib matplotlib
import argparse
import csv
import glob
import json
import math
import os
import warnings

import numpy as np
import pyedflib
from scipy import stats
from scipy.signal import butter, filtfilt, find_peaks
from sklearn.decomposition import FastICA
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


# ═══════════════════════════════════════════════════════════════════════
# НАСТРОЙКИ
# ═══════════════════════════════════════════════════════════════════════

LEAD_II_CHANNEL = 1       # номер канала отведения II в записи (каналы 0, 1, 2 = отведения I, II, III)
# Лучшее отведение каждой записи — то, в котором зубцы P, Q, R, S, T определяются
# надёжнее всего (критерий чистоты разметки, п. 7): 0 = I, 1 = II, 2 = III.
# Для новых записей: запустите код П.7 и перенесите сюда столбец «выбрано».
P7_LEADS = {'0011': 1, '0012': 1, '0013': 1, '0014': 1, '0015': 1, '0016': 0, '0017': 2,
            '0018': 1, '0019': 1, '0020': 2, '0021': 1, '0022': 1, '0023': 1, '0024': 1,
            '0025': 2, '0026': 1, '0027': 1, '0028': 0, '0029': 1, '0030': 1}
LEAD_NAMES = {0: 'I', 1: 'II', 2: 'III'}
MIN_N_WHOLE = 300         # минимум годных циклов для расчёта по записи целиком
MIN_N_WHOLE_SOFT = 250    # смягчённый порог (проверка устойчивости)
MIN_N_PHASE = 150         # минимум годных циклов внутри одной фазы
W_SLOW = 15               # окно скользящего среднего, циклов
HALF = W_SLOW // 2
REST_WINDOW = (30.0, -5.0)    # покой: от 30 с до (начало подъёма − 5 с)
TILT_WINDOW = (15.0, 275.0)   # наклон: от (конец подъёма + 15 с) до (конец подъёма + 275 с)
ICA_SEED = 0


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 1. Чтение записи и детекторы R, P, T (общие для всех пунктов);
#          границы фаз — по протоколу пробы
# ═══════════════════════════════════════════════════════════════════════

def read_ecg(path, ch=0):
    f = pyedflib.EdfReader(path)
    fs = float(f.getSampleFrequency(ch))
    sig = f.readSignal(ch).astype(float)
    f.close()
    return sig, fs


def detect_r(sig, fs, band=(5.0, 20.0), integ_ms=120,
             min_rr_ms=300, win_s=10.0, k=0.35, refine_ms=100):
    x = np.asarray(sig, float)
    x = x - np.median(x)

    # 1) полосовой фильтр — полоса энергии QRS
    ny = fs/2
    b, a = butter(3, [band[0]/ny, min(band[1]/ny, 0.99)], btype='band')
    xf = filtfilt(b, a, x)

    # 2) производная + возведение в квадрат (подчёркивает крутые фронты QRS)
    d = np.diff(xf, append=xf[-1])
    sq = d**2

    # 3) интегрирование скользящим окном ~ длительность QRS
    w = max(3, int(round(integ_ms/1000*fs)))
    integ = np.convolve(sq, np.ones(w)/w, mode='same')

    # 4) ЛОКАЛЬНО адаптивный порог: доля от локального максимума
    #    (окно win_s секунд, шаг — половина окна)
    n = len(integ)
    W = max(int(win_s*fs), w*4)
    thr = np.zeros(n)
    for start in range(0, n, W//2):
        end = min(start+W, n)
        seg = integ[start:end]
        if len(seg) == 0:
            continue
        local = k*np.percentile(seg, 98)      # 98-й процентиль ≈ уровень QRS
        thr[start:end] = np.maximum(thr[start:end], local)
    thr[thr == 0] = k*np.percentile(integ, 98)

    # 5) поиск пиков с рефрактерным периодом
    dist = max(1, int(round(min_rr_ms/1000*fs)))
    cand, _ = find_peaks(integ, height=thr, distance=dist)

    # 6) уточнение положения по максимуму |полосового сигнала|
    half = max(2, int(round(refine_ms/1000*fs/2)))
    out = []
    for c in cand:
        a0, b0 = max(0, c-half), min(len(xf), c+half+1)
        out.append(a0 + int(np.argmax(np.abs(xf[a0:b0]))))
    return np.array(sorted(set(out)), dtype=int), xf


def suppress_qrs(sig, r_idx, fs, before_ms=50, after_ms=100):
    out = sig.copy()
    b = int(round(before_ms/1000*fs)); a = int(round(after_ms/1000*fs))
    for r in r_idx:
        i0, i1 = max(0, r-b), min(len(sig)-1, r+a)
        if i1-i0 < 2: continue
        out[i0:i1+1] = np.interp(np.arange(i0, i1+1), [i0, i1], [sig[i0], sig[i1]])
    return out


def halfwave(lam):
    """Вейвлет «полуволна»: положительный полупериод синуса
       с симметричными отрицательными крыльями (нулевое среднее)."""
    if lam % 2 == 0: lam += 1
    pos = np.sin(np.linspace(0, math.pi, lam))
    area = float(np.trapezoid(pos)) if hasattr(np, 'trapezoid') else float(np.trapz(pos))
    wing = max(1, lam//4)
    w = np.zeros(lam + 2*wing)
    w[:wing] = -area/(2*wing)
    w[wing:wing+lam] = pos
    w[wing+lam:] = -area/(2*wing)
    return w - w.mean()


def detect_pt(sig, r_idx, fs, wave_ms=85, thr_frac=0.3, local_s=30.0, noise_k=10.0,
              tol_ms=20.0, win=21):
    """Возвращает индексы P и T (по одному на кардиоцикл, если найдены).
    Отклик — свёртка ЭКГ (с подавленными комплексами QRS) с вейвлетом «полуволна».
    T и основной поиск P: порог = thr_frac · 99-й процентиль положительного
    отклика по всей записи (без участков с аномально длинными RR).
    Дополнительный поиск P (для записей, где P в одной из фаз заметно меньше):
    порог по откликам в окнах поиска P в скользящем окне ±local_s секунд, но не
    ниже noise_k · уровня шума; такая находка принимается, только если расстояние
    P–R отличается от скользящей медианы по win циклам не больше tol_ms."""
    r_idx = np.asarray(r_idx, int)
    ecg = suppress_qrs(sig, r_idx, fs)
    lam = max(5, int(round(wave_ms/1000*fs)))
    w = halfwave(lam)
    conv = np.convolve(ecg, w[::-1], mode='same')
    n = len(conv)

    # порог по «здоровым» участкам (исключаем аномально длинные RR)
    rr = np.diff(r_idx)
    med = np.median(rr) if len(rr) else 0
    ok = np.ones(n, bool)
    for k in range(len(rr)):
        if med and rr[k] > 1.8*med:
            ok[r_idx[k]:r_idx[k+1]] = False
    pos = conv[(conv > 0) & ok]
    thr = thr_frac*np.percentile(pos, 99) if len(pos) else 0

    # окна поиска P и T
    wins = []
    for k in range(len(r_idx)):
        r = r_idx[k]
        r_prev = r_idx[k-1] if k > 0 else max(0, r-int(fs))
        r_next = r_idx[k+1] if k < len(r_idx)-1 else min(len(sig)-1, r+int(fs))
        # P: непосредственно перед QRS (P предшествует R на 100–300 мс)
        rr_p = r - r_prev
        lo = max(r_prev + rr_p//2, r - int(0.35*fs))
        ps, pe = lo, r - max(1, int(0.06*fs))
        # T: после QRS, до начала следующего цикла
        rr_n = r_next - r
        ts = r + max(int(0.12*fs), rr_n//8)
        te = min(r + int(0.6*rr_n), r_next - int(0.08*fs))
        wins.append((ps, pe, ts, te))

    P, T = [], []
    for ps, pe, ts, te in wins:
        if ps < pe <= len(conv):
            seg = conv[ps:pe]
            P.append(ps + int(np.argmax(seg)) if len(seg) and seg.max() > thr else -1)
        else:
            P.append(-1)
        if ts < te <= len(conv):
            seg = conv[ts:te]
            T.append(ts + int(np.argmax(seg)) if len(seg) and seg.max() > thr else -1)
        else:
            T.append(-1)
    P = np.array(P); T = np.array(T)

    # дополнительный поиск P: порог по окнам P в скользящем окне со страховкой от шума
    pmask = np.zeros(n, bool)
    for ps, pe, _, _ in wins:
        if ps < pe <= n:
            pmask[ps:pe] = True
    d = np.diff(ecg)
    noise = 1.4826*np.median(np.abs(d - np.median(d)))/np.sqrt(2)*np.sqrt(np.sum(w**2))
    L = int(local_s*fs)
    P2 = np.full(len(r_idx), -1)
    for k, (ps, pe, _, _) in enumerate(wins):
        if not (ps < pe <= n):
            continue
        i = r_idx[k]; a, b = max(0, i-L), min(n, i+L)
        sg = conv[a:b]; mk = (sg > 0) & ok[a:b] & pmask[a:b]
        thr_p = thr_frac*np.percentile(sg[mk], 99) if mk.sum() > 10 else thr
        seg = conv[ps:pe]
        if len(seg) and seg.max() > max(thr_p, noise_k*noise):
            P2[k] = ps + int(np.argmax(seg))
    off = np.where(P2 >= 0, (r_idx - P2)/fs*1000, np.nan)
    med_off = np.full(len(off), np.nan); h = win//2
    for i in range(len(off)):
        s = off[max(0, i-h):i+h+1]; s = s[np.isfinite(s)]
        if len(s) >= 3:
            med_off[i] = np.median(s)
    add = (P < 0) & (P2 >= 0) & (np.abs(off - med_off) <= tol_ms)
    P[add] = P2[add]
    return P, T, ecg, conv


def phase_bounds(marks):
    """Границы подъёма стола по протоколу пробы. Метки «Этап»: около 05:00 —
    начало подъёма, около 05:10–05:20 — конец подъёма, около 10:00 — конец
    записи. Если в зоне подъёма несколько меток, первая — начало, последняя —
    конец; если одна и она позже 05:10 — это конец подъёма, начало принимается
    на 15 с раньше. Возвращает (начало подъёма, конец подъёма), с."""
    mid = sorted(t for t in marks if 250 < t < 340)
    if len(mid) >= 2:
        return mid[0], mid[-1]
    if len(mid) == 1:
        m = mid[0]
        return (m - 15.0, m) if m >= 310 else (m, m + 15.0)
    return 300.0, 315.0

# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 2. Ряды RR, PP, PR (с уточнением положений R и P)
# ═══════════════════════════════════════════════════════════════════════

def band_ecg(sig,fs,lo=0.5,hi=40.0):
    b,a=butter(2,[lo/(fs/2),hi/(fs/2)],'band'); return filtfilt(b,a,sig)

def parabolic(y,i):
    """Вершина параболы через три точки вокруг i: поправка в долях отсчёта."""
    if i<=0 or i>=len(y)-1: return float(i)
    a,b,c=y[i-1],y[i],y[i+1]; den=a-2*b+c
    return float(i) if den==0 else i+0.5*(a-c)/den

def refine_r(y,r,fs,half_ms=8):
    """Уточнение вершины R: максимум |ЭКГ| рядом с отметкой + парабола."""
    h=int(half_ms/1000*fs); out=np.full(len(r),np.nan)
    for k,ri in enumerate(r):
        a,b=max(1,ri-h),min(len(y)-1,ri+h+1); seg=np.abs(y[a:b]); i=int(np.argmax(seg))
        out[k]=a+parabolic(seg,i)
    return out

def refine_p(y,P,fs,half_ms=60,search_ms=20,iters=2):
    """Уточнение положения P по шаблону: шаблон — среднее участков ЭКГ вокруг
    найденных P; в каждом цикле — сдвиг максимальной корреляции с шаблоном
    (в пределах ±search_ms) с параболической интерполяцией."""
    P=np.asarray(P); h=int(half_ms/1000*fs); s=int(search_ms/1000*fs); idx=np.where(P>=0)[0]
    pos=P.astype(float).copy(); pos[P<0]=np.nan
    for _ in range(iters):
        segs=[]
        for k in idx:
            c=int(round(pos[k]))
            if c-h-s>=0 and c+h+s<len(y): segs.append(y[c-h:c+h+1]-np.mean(y[c-h:c+h+1]))
        if len(segs)<20: return pos
        tpl=np.mean(segs,0); tpl=tpl-tpl.mean(); tn=np.linalg.norm(tpl)
        new=pos.copy()
        for k in idx:
            c=int(round(pos[k]))
            if not (c-h-s>=0 and c+h+s<len(y)): continue
            cc=np.empty(2*s+1)
            for j,l in enumerate(range(-s,s+1)):
                w=y[c+l-h:c+l+h+1]; w=w-w.mean(); nw=np.linalg.norm(w)
                cc[j]=np.dot(w,tpl)/(nw*tn) if nw>0 else -1
            i=int(np.argmax(cc)); new[k]=c+(parabolic(cc,i)-s)
        pos=new
    return pos


def build_series_ch(path, ch, refine=True):
    """Ряды RR, PP и PR одного отведения. Положения R и P уточняются точнее
    одного отсчёта: R — параболической интерполяцией вершины, P — по шаблону
    зубца P записи (корреляция в пределах ±20 мс с интерполяцией). Все три ряда
    строятся из одних и тех же уточнённых точек, поэтому тождество
    RR(k) = PP(k) + PR(k+1) − PR(k) выполняется точно."""
    f = pyedflib.EdfReader(path)
    ann = f.readAnnotations()
    f.close()
    marks = [float(t) for t in ann[0]]
    p1e, p2b = phase_bounds(marks)

    sig, fs = read_ecg(path, ch=ch)
    r, _ = detect_r(sig, fs)
    P, T, _, _ = detect_pt(sig, r, fs)
    if refine:
        y = band_ecg(sig, fs)
        r_ref = refine_r(y, r, fs)
        p_ref = refine_p(y, P, fs)
    else:                                   # вариант без уточнения: точки — индексы отсчётов
        r_ref = r.astype(float)
        p_ref = np.where(P >= 0, P, np.nan).astype(float)

    t_r = r_ref/fs
    rr = np.diff(r_ref)/fs*1000
    has_p = (P > 0) & np.isfinite(p_ref)
    pr = np.full(len(r), np.nan)
    pr[has_p] = (r_ref[has_p] - p_ref[has_p])/fs*1000
    pp = np.full(len(r), np.nan)
    idx = np.where(has_p)[0]
    for a, b in zip(idx[:-1], idx[1:]):
        if b == a + 1:
            pp[a] = (p_ref[b] - p_ref[a])/fs*1000

    return dict(t_r=t_r, rr=rr, t_rr=t_r[:-1], pr=pr, pp=pp,
                p1e=p1e, p2b=p2b, n_r=len(r),
                n_pr=int(np.isfinite(pr).sum()), n_pp=int(np.isfinite(pp).sum()))




# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 3. ICA и регрессия на паре рядов
# ═══════════════════════════════════════════════════════════════════════

def zscore(x):
    return (x - x.mean()) / x.std(ddof=1)


def fast_part(x):
    """Быстрая часть ряда: ряд минус скользящее среднее по W_SLOW циклам
    (края по HALF циклов отбрасываются)."""
    return x[HALF:-HALF] - np.convolve(x, np.ones(W_SLOW) / W_SLOW, mode='valid')


def ica_vs_ols(A, B, target=None, seed=ICA_SEED):
    """Пара рядов: A — «ритм», B — ряд, из которого извлекается собственная
    составляющая. Возвращает |r| остатка ICA и остатка регрессии B на A
    с целью, их сходство и |r| ритмической компоненты ICA с A.
    target=None — цель: быстрая часть B;
    иначе — заданный ряд той же длины (контрольная пара PP+RR)."""
    a, b = np.polyfit(A, B, 1)
    ols = B - (a * A + b)
    S = FastICA(n_components=2, whiten='unit-variance', random_state=seed,
                max_iter=2000).fit_transform(np.column_stack([zscore(A), zscore(B)]))
    corr_to_A = [abs(np.corrcoef(S[:, k], A)[0, 1]) for k in (0, 1)]
    k_rate = int(np.argmax(corr_to_A))
    ica = S[:, 1 - k_rate]
    r_rate = corr_to_A[k_rate]
    if target is None:
        t = fast_part(B); ica, ols = ica[HALF:-HALF], ols[HALF:-HALF]
    else:
        t = target
    return dict(r_ica=abs(np.corrcoef(ica, t)[0, 1]), r_ols=abs(np.corrcoef(ols, t)[0, 1]),
                r_sim=abs(np.corrcoef(ica, ols)[0, 1]), r_rate=float(r_rate))


def ica_vs_ols_highpass(A, B, seed=ICA_SEED):
    """Вариант «без медленной части»: оба ряда заменяются их быстрой частью,
    цель — быстрая часть B."""
    Ah, Bh = fast_part(A), fast_part(B)
    a, b = np.polyfit(Ah, Bh, 1)
    ols = Bh - (a * Ah + b)
    S = FastICA(n_components=2, whiten='unit-variance', random_state=seed,
                max_iter=2000).fit_transform(np.column_stack([zscore(Ah), zscore(Bh)]))
    corr_to_A = [abs(np.corrcoef(S[:, k], Ah)[0, 1]) for k in (0, 1)]
    k_rate = int(np.argmax(corr_to_A)); ica = S[:, 1 - k_rate]
    return dict(r_ica=abs(np.corrcoef(ica, Bh)[0, 1]), r_ols=abs(np.corrcoef(ols, Bh)[0, 1]),
                r_sim=abs(np.corrcoef(ica, ols)[0, 1]), r_rate=float(corr_to_A[k_rate]))


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 4. Варианты расчёта по записям
# ═══════════════════════════════════════════════════════════════════════

def pair_arrays(v):
    """Выровненные ряды одной записи: RR, PP, PR, разность соседних PR и время."""
    n = min(len(v['rr']), len(v['pp']), len(v['pr']) - 1)
    return dict(RR=v['rr'][:n], PP=v['pp'][:n], PR=v['pr'][:n],
                dPR=v['pr'][1:n + 1] - v['pr'][:n], t=v['t_r'][:n])


def phase_mask(v, t, phase):
    if phase == 'покой':
        a, b = REST_WINDOW[0], v['p1e'] + REST_WINDOW[1]
    else:
        a, b = v['p2b'] + TILT_WINDOW[0], v['p2b'] + TILT_WINDOW[1]
    return (t >= a) & (t <= b)


def outlier_mask(x, k=4.0):
    """Выбросы быстрой части ряда: отклонение от медианы больше k·MAD."""
    f = fast_part(x); med = np.median(f); mad = 1.4826 * np.median(np.abs(f - med))
    m = np.zeros(len(x), bool); m[HALF:-HALF] = np.abs(f - med) > k * mad
    return m


VARIANTS = ['целиком', 'целиком, порог 250', 'без выбросов', 'покой', 'наклон',
            'покой без медленной части', 'наклон без медленной части']


def variant_mask(v, s, A, B, variant):
    ok = np.isfinite(s[A]) & np.isfinite(s[B])
    if B == 'RR':
        ok &= np.isfinite(s['dPR'])
    if variant in ('целиком', 'целиком, порог 250'):
        return ok, (MIN_N_WHOLE_SOFT if 'порог' in variant else MIN_N_WHOLE)
    if variant == 'без выбросов':
        m = ok.copy(); idx = np.where(ok)[0]
        if len(idx) >= MIN_N_WHOLE:
            bad = outlier_mask(s[A][idx]) | outlier_mask(s[B][idx]); m[idx[bad]] = False
        return m, MIN_N_WHOLE
    phase = 'покой' if variant.startswith('покой') else 'наклон'
    return ok & phase_mask(v, s['t'], phase), MIN_N_PHASE


def analyse(series, A, B, variant, seed=ICA_SEED):
    """Расчёт по всем записям. Возвращает строки по каждой записи (с причиной
    исключения) и сводку."""
    rows = []
    for name, v in sorted(series.items()):
        s = pair_arrays(v); m, min_n = variant_mask(v, s, A, B, variant); k = int(m.sum())
        if k < min_n:
            rows.append(dict(record=name, cycles=k, used=False, reason=f'годных циклов {k} < порога {min_n}'))
            continue
        if variant.endswith('без медленной части'):
            r = ica_vs_ols_highpass(s[A][m], s[B][m], seed)
        elif B == 'RR':
            r = ica_vs_ols(s[A][m], s[B][m], target=s['dPR'][m], seed=seed)
        else:
            r = ica_vs_ols(s[A][m], s[B][m], seed=seed)
        rows.append(dict(record=name, cycles=k, used=True, reason='', **r))
    used = [r for r in rows if r['used']]
    ri = np.array([r['r_ica'] for r in used]); ro = np.array([r['r_ols'] for r in used])
    summ = dict(n=len(used), med_ica=float(np.median(ri)), med_ols=float(np.median(ro)),
                gap=float(np.median(ro - ri)), better=int((ri > ro).sum()),
                p=float(stats.wilcoxon(ri - ro)[1]),
                med_sim=float(np.median([r['r_sim'] for r in used])),
                med_rate=float(np.median([r['r_rate'] for r in used])))
    return rows, summ


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 5. Свойства рядов: тождество, негауссовость, смена фазы, зубец P
# ═══════════════════════════════════════════════════════════════════════

def identity_check(series):
    """Максимальное отклонение от тождества RR(k) = PP(k) + PR(k+1) − PR(k), мс."""
    worst, n = 0.0, 0
    for v in series.values():
        s = pair_arrays(v); rhs = s['PP'] + s['dPR']; ok = np.isfinite(rhs) & np.isfinite(s['RR'])
        if ok.any():
            worst = max(worst, float(np.max(np.abs(s['RR'][ok] - rhs[ok])))); n += int(ok.sum())
    return dict(max_abs_ms=worst, cycles=n)


def series_properties(series):
    """Доля разброса от смены фазы и эксцесс рядов (целиком и внутри фаз)."""
    out = {k: dict(step=[], kurt_full=[], kurt_rest=[], kurt_tilt=[]) for k in ('PP', 'PR', 'RR')}
    for v in series.values():
        s = pair_arrays(v); ok = np.isfinite(s['PP']) & np.isfinite(s['PR']) & np.isfinite(s['RR'])
        if ok.sum() < MIN_N_WHOLE:
            continue
        rest = phase_mask(v, s['t'], 'покой'); tilt = phase_mask(v, s['t'], 'наклон')
        for k in out:
            x = s[k]; inph = ok & (rest | tilt); xm = x[inph]; g = tilt[inph].astype(int)
            ss_t = ((xm - xm.mean()) ** 2).sum()
            ss_b = sum(((xm[g == j].mean() - xm.mean()) ** 2) * (g == j).sum() for j in (0, 1))
            out[k]['step'].append(ss_b / ss_t)
            out[k]['kurt_full'].append(stats.kurtosis(x[ok]))
            out[k]['kurt_rest'].append(stats.kurtosis(x[ok & rest]) if (ok & rest).sum() > 50 else np.nan)
            out[k]['kurt_tilt'].append(stats.kurtosis(x[ok & tilt]) if (ok & tilt).sum() > 50 else np.nan)
    return {k: {f: float(np.nanmedian(vals)) for f, vals in d.items()} | dict(records=len(d['step']))
            for k, d in out.items()}


def p_detection_by_phase(series):
    """Доля циклов с найденным зубцом P в покое и при наклоне по каждой записи."""
    out = {}
    for name, v in sorted(series.items()):
        t = v['t_r']; has = np.isfinite(v['pr'])
        rest = phase_mask(v, t, 'покой'); tilt = phase_mask(v, t, 'наклон')
        out[name] = dict(rest=float(has[rest].mean()), tilt=float(has[tilt].mean()))
    return out


def ica_robustness(series_by_lead, keys, seeds=range(20)):
    """Устойчивость вывода к случайному старту FastICA и к настройкам алгоритма.
    keys — список (отведения, A, B, вариант)."""
    global FastICA
    out = {}
    for lead, A, B, variant in keys:
        g, p = [], []
        for seed in seeds:
            _, sm = analyse(series_by_lead[lead], A, B, variant, seed=seed); g.append(sm['gap']); p.append(sm['p'])
        g, p = np.array(g), np.array(p)
        out[f'{lead}|{A}+{B}|{variant}|старты'] = dict(
            starts=len(g), gap_min=float(g.min()), gap_max=float(g.max()), p_min=float(p.min()), p_max=float(p.max()),
            worse_and_significant=int(((g > 0) & (p < 0.05)).sum()), worse=int((g > 0).sum()))
    base = FastICA
    try:
        for fun, alg in [('logcosh', 'parallel'), ('exp', 'parallel'), ('cube', 'parallel'), ('logcosh', 'deflation')]:
            FastICA = (lambda fun_, alg_: (lambda **kw: base(fun=fun_, algorithm=alg_, **kw)))(fun, alg)
            for lead, A, B, variant in keys:
                if variant != 'целиком':
                    continue
                _, sm = analyse(series_by_lead[lead], A, B, variant)
                out[f'{lead}|{A}+{B}|{variant}|{fun}, {alg}'] = dict(gap=sm['gap'], p=sm['p'])
    finally:
        FastICA = base
    return out


def slow_share(series):
    """Доля медленной части (скользящее среднее по W_SLOW циклам) в разбросе
    каждого ряда внутри фазы; медиана по записям, где в фазе не меньше
    MIN_N_PHASE годных циклов."""
    out = {}
    for phase in ('покой', 'наклон'):
        for k in ('PP', 'PR', 'RR'):
            vals = []
            for v in series.values():
                s = pair_arrays(v); m = np.isfinite(s[k]) & phase_mask(v, s['t'], phase)
                if m.sum() < MIN_N_PHASE:
                    continue
                x = s[k][m]; sl = np.convolve(x, np.ones(W_SLOW) / W_SLOW, mode='valid')
                vals.append(np.var(sl - sl.mean()) / np.var(x[HALF:-HALF]))
            out[f'{phase}|{k}'] = dict(median=float(np.median(vals)), records=len(vals))
    return out


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 6. Синтетика: когда ICA вообще может обогнать регрессию
# ═══════════════════════════════════════════════════════════════════════

def _ar1(n, phi, rng):
    e = rng.standard_normal(n); x = np.zeros(n)
    for i in range(1, n):
        x[i] = phi * x[i - 1] + e[i]
    return zscore(x)


def synth_pair(rng, n=700, a=0.15, s_sd=3.0, u_sd=0.0, gaussian=False, noise_ms=1.0, quant=2.0):
    """PP = c + u + шум; PR = a·c + s + шум. c — общий модулятор ритма
    (медленная волна, дыхательная волна, медленный шум), s — собственная
    составляющая PR, u — собственная составляющая PP. Квантование 2 мс
    соответствует частоте дискретизации 500 Гц."""
    k = np.arange(n)
    c = zscore(np.sin(2 * np.pi * k / 40) + 0.8 * np.sin(2 * np.pi * k / 4.5 + rng.uniform(0, 6))
               + 0.7 * _ar1(n, 0.9, rng)) * 40
    s = (_ar1(n, 0.3, rng) if gaussian else zscore(rng.laplace(size=n))) * s_sd
    u = (_ar1(n, 0.3, rng) if gaussian else zscore(rng.laplace(size=n))) * u_sd
    q = (lambda x: np.round(x / quant) * quant) if quant else (lambda x: x)
    return q(800 + c + u + noise_ms * rng.standard_normal(n)), q(150 + a * c + s + noise_ms * rng.standard_normal(n)), s


def _resid(A, B, seed=ICA_SEED):
    a, b = np.polyfit(A, B, 1); ols = B - (a * A + b)
    S = FastICA(n_components=2, whiten='unit-variance', random_state=seed,
                max_iter=2000).fit_transform(np.column_stack([zscore(A), zscore(B)]))
    k = int(np.argmax([abs(np.corrcoef(S[:, j], A)[0, 1]) for j in (0, 1)]))
    return ols, S[:, 1 - k]


def synthetic_checks(reps=40):
    """Синтетические сценарии. Для каждого: медианы |r| остатков с ИСТИННОЙ
    собственной составляющей PR, доля повторов, где ICA лучше, и |r| быстрой
    части PR с истинной составляющей (насколько годится мерило отчёта)."""
    def scen(**kw):
        rng = np.random.default_rng(1); R = []
        for _ in range(reps):
            pp, pr, s = synth_pair(rng, **kw); ols, ica = _resid(pp, pr)
            R.append([abs(np.corrcoef(ols, s)[0, 1]), abs(np.corrcoef(ica, s)[0, 1]),
                      abs(np.corrcoef(fast_part(pr), s[HALF:-HALF])[0, 1])])
        R = np.array(R)
        return dict(ols=float(np.median(R[:, 0])), ica=float(np.median(R[:, 1])),
                    ica_wins=float(np.mean(R[:, 1] > R[:, 0])), yardstick=float(np.median(R[:, 2])))
    out = {'базовый: у PP своей составляющей нет': scen(),
           'собственная составляющая PR гауссова': scen(gaussian=True),
           'слабая связь PR с ритмом (a = 0,05)': scen(a=0.05),
           'сильная связь PR с ритмом (a = 0,3)': scen(a=0.3),
           'у PP своя составляющая (3 источника на 2 ряда)': scen(u_sd=20),
           'то же, источники гауссовы': scen(u_sd=20, gaussian=True)}
    # контроль: два негауссовых источника, оба входят в оба ряда — ICA обязана выиграть
    rng = np.random.default_rng(3); R = []
    for _ in range(reps):
        c = zscore(rng.laplace(size=700)) * 40; s = zscore(rng.laplace(size=700)) * 3
        pp = 800 + c + 4 * s; pr = 150 + 0.15 * c + s + 0.3 * rng.standard_normal(700)
        ols, ica = _resid(pp, pr); R.append([abs(np.corrcoef(ols, s)[0, 1]), abs(np.corrcoef(ica, s)[0, 1])])
    R = np.array(R)
    out['контроль: 2 негауссовых источника в обоих рядах'] = dict(
        ols=float(np.median(R[:, 0])), ica=float(np.median(R[:, 1])), ica_wins=float(np.mean(R[:, 1] > R[:, 0])),
        yardstick=float('nan'))
    # проверка мер негауссовости и квантования
    rng = np.random.default_rng(0)
    out['_меры'] = dict(gauss=float(stats.kurtosis(rng.normal(size=700))), laplace=float(stats.kurtosis(rng.laplace(size=700))),
                        uniform=float(stats.kurtosis(rng.uniform(-1, 1, 700))),
                        quantized_gauss=float(np.median([stats.kurtosis(fast_part(np.round(rng.normal(150, 2.5, 300) / 2) * 2))
                                                         for _ in range(200)])),
                        step=float(stats.kurtosis(np.r_[np.zeros(350), np.ones(350)] * 4
                                                  + np.random.default_rng(7).normal(size=700))))
    return out


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 7. Рисунки
# ═══════════════════════════════════════════════════════════════════════

def _comma_axes(fig):
    """Десятичная запятая на линейных осях (русская запись чисел)."""
    from matplotlib.ticker import FuncFormatter
    fmt = FuncFormatter(lambda v, _: f'{v:g}'.replace('.', ','))
    for a in fig.axes:
        if a.get_yscale() == 'linear':
            a.yaxis.set_major_formatter(fmt)


def fig_records(rows_ecg2, rows_p7, path):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
    for ax, rows, title in [(axes[0], rows_ecg2, 'а — отведение II'), (axes[1], rows_p7, 'б — лучшее отведение каждой записи')]:
        used = [r for r in rows if r['used']]; x = np.arange(len(used)); w = 0.38
        ax.bar(x - w / 2, [r['r_ica'] for r in used], w, color='#c62828', label='остаток ICA')
        ax.bar(x + w / 2, [r['r_ols'] for r in used], w, color='#1565c0', label='остаток регрессии')
        ax.set_xticks(x); ax.set_xticklabels([r['record'] for r in used], rotation=90, fontsize=8)
        ax.set_ylim(0, 1.05); ax.set_title(title, fontsize=11); ax.grid(alpha=0.3, axis='y')
    axes[0].set_ylabel('|r| с быстрой частью PR'); axes[0].legend(fontsize=9)
    _comma_axes(fig); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def fig_phases(results, path):
    labels, before, after = [], [], []
    for lead in ('II', 'лучшее'):
        for pair in ('PP+PR', 'RR+PR'):
            for ph in ('покой', 'наклон'):
                labels.append(f'{pair}\n{lead}\n{ph}')
                before.append(results[(lead, pair, ph)]['summary']['gap'])
                after.append(results[(lead, pair, ph + ' без медленной части')]['summary']['gap'])
    x = np.arange(len(labels)); w = 0.38
    fig, ax = plt.subplots(figsize=(13, 4.6))
    ax.bar(x - w / 2, before, w, color='#ef6c00', label='исходные ряды')
    ax.bar(x + w / 2, after, w, color='#2e7d32', label='без медленной части')
    ax.axhline(0, color='k', lw=0.8); ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('разрыв: медиана разностей |r| (регрессия − ICA)'); ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')
    _comma_axes(fig); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def fig_pdetect(pdet, path):
    names = sorted(pdet); x = np.arange(len(names)); w = 0.38
    fig, ax = plt.subplots(figsize=(13, 4.2))
    ax.bar(x - w / 2, [pdet[n]['rest'] * 100 for n in names], w, color='#1565c0', label='покой')
    ax.bar(x + w / 2, [pdet[n]['tilt'] * 100 for n in names], w, color='#c62828', label='наклон')
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=90, fontsize=8); ax.set_ylim(0, 105)
    ax.set_ylabel('циклов с найденным P, %'); ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')
    _comma_axes(fig); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 8. Общий запуск
# ═══════════════════════════════════════════════════════════════════════

PAIRS = [('PP', 'PR'), ('RR', 'PR'), ('PP', 'RR')]


def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    return o


def run_p3(data_dir, out_dir='results_p3', with_synthetic=True):
    os.makedirs(out_dir, exist_ok=True)
    paths = sorted(glob.glob(os.path.join(data_dir, '00*_ecg.edf')))
    if not paths:
        raise SystemExit(f'В папке {data_dir} нет файлов 00*_ecg.edf')
    names = [os.path.basename(p)[:4] for p in paths]
    missing = [n for n in names if n not in P7_LEADS]
    if missing:
        raise SystemExit(f'Нет отведения П.7 для записей {missing}: дополните таблицу P7_LEADS в НАСТРОЙКАХ')
    print(f'Записей: {len(paths)}. Лучшее отведение по записям: ' +
          ', '.join(f'{n} — {LEAD_NAMES[P7_LEADS[n]]}' for n in names))
    print('Строю ряды RR, PP, PR...')
    leadsets = {'II': {n: build_series_ch(p, LEAD_II_CHANNEL) for n, p in zip(names, paths)},
                'лучшее': {n: build_series_ch(p, P7_LEADS[n]) for n, p in zip(names, paths)}}
    summary = {'identity': identity_check(leadsets['II'])}
    print('Вариант без уточнения положений R и P (отведение II)...')
    raw_ii = {n: build_series_ch(p, LEAD_II_CHANNEL, refine=False) for n, p in zip(names, paths)}
    summary['unrefined_ii'] = {f'{A}+{B}': analyse(raw_ii, A, B, 'целиком')[1] for A, B in (('PP', 'PR'), ('RR', 'PR'))}
    for k, sm in summary['unrefined_ii'].items():
        print(f"  без уточнения, {k}: ICA {sm['med_ica']:.3f}, регрессия {sm['med_ols']:.3f}, ICA лучше {sm['better']} из {sm['n']}, p = {sm['p']:.4f}")
    print(f"Тождество RR(k) = PP(k) + PR(k+1) − PR(k): максимальное отклонение "
          f"{summary['identity']['max_abs_ms']:.3g} мс по {summary['identity']['cycles']} циклам")

    results = {}
    print('\nсочетание                                   записей   ICA    регр.  разрыв  ICA лучше      p')
    for lead in ('II', 'лучшее'):
        for A, B in PAIRS:
            for variant in VARIANTS:
                if (A, B) == ('PP', 'RR') and variant.endswith('медленной части'):
                    continue
                rows, sm = analyse(leadsets[lead], A, B, variant)
                results[(lead, f'{A}+{B}', variant)] = dict(rows=rows, summary=sm)
                print(f"{lead:4s} {A}+{B:3s} {variant:27s} {sm['n']:6d}  {sm['med_ica']:.3f}  {sm['med_ols']:.3f}  "
                      f"{sm['gap']:+.3f}  {sm['better']:3d} из {sm['n']:<3d} {sm['p']:.4f}")
    main = results[('II', 'PP+PR', 'целиком')]['summary']
    print(f"\nОсновной расчёт (отведение II, пара PP + PR, запись целиком): записей {main['n']}, медиана |r| ICA {main['med_ica']:.3f}, "
          f"регрессии {main['med_ols']:.3f}, ICA лучше в {main['better']} из {main['n']}, p = {main['p']:.4f}; "
          f"|r(IC-ритм, PP)| {main['med_rate']:.3f}; |r(остаток ICA, остаток регрессии)| {main['med_sim']:.3f}")

    print('\nУстойчивость к случайному старту ICA (20 стартов) и к настройкам алгоритма...')
    rob_keys = [('II', 'PP', 'PR', 'целиком'), ('лучшее', 'PP', 'PR', 'целиком'), ('II', 'RR', 'PR', 'целиком'),
                ('лучшее', 'RR', 'PR', 'целиком'), ('лучшее', 'PP', 'PR', 'наклон'), ('лучшее', 'PP', 'PR', 'покой'),
                ('II', 'PP', 'PR', 'без выбросов'), ('лучшее', 'PP', 'PR', 'без выбросов')]
    summary['robustness'] = ica_robustness(leadsets, rob_keys)
    for k, v in summary['robustness'].items():
        if k.endswith('старты'):
            print(f"  {k:32s} разрыв от {v['gap_min']:+.3f} до {v['gap_max']:+.3f}; p от {v['p_min']:.4f} до {v['p_max']:.4f}; "
                  f"ICA хуже и p<0,05 при {v['worse_and_significant']} из {v['starts']} стартов")
        else:
            print(f"  {k:32s} разрыв {v['gap']:+.3f}, p = {v['p']:.4f}")

    summary['properties'] = {lead: series_properties(leadsets[lead]) for lead in leadsets}
    summary['slow_share'] = {lead: slow_share(leadsets[lead]) for lead in leadsets}
    for lead in leadsets:
        ss = summary['slow_share'][lead]
        print(f"Доля медленной части ({lead}): покой PP {ss['покой|PP']['median']:.0%} PR {ss['покой|PR']['median']:.0%} "
              f"RR {ss['покой|RR']['median']:.0%}; наклон PP {ss['наклон|PP']['median']:.0%} PR {ss['наклон|PR']['median']:.0%} "
              f"RR {ss['наклон|RR']['median']:.0%}")
    summary['p_detection_lead_ii'] = p_detection_by_phase(leadsets['II'])
    if with_synthetic:
        print('\nСинтетическая проверка...')
        summary['synthetic'] = synthetic_checks()
        for k, v in summary['synthetic'].items():
            if not k.startswith('_'):
                print(f"  {k:50s} регр {v['ols']:.2f}  ICA {v['ica']:.2f}  ICA лучше в {v['ica_wins'] * 100:3.0f}% повторов")

    fig_records(results[('II', 'PP+PR', 'целиком')]['rows'], results[('лучшее', 'PP+PR', 'целиком')]['rows'],
                os.path.join(out_dir, 'p3_fig1_records.png'))
    fig_phases(results, os.path.join(out_dir, 'p3_fig2_phases.png'))
    fig_pdetect(summary['p_detection_lead_ii'], os.path.join(out_dir, 'p3_fig3_p_detection.png'))

    with open(os.path.join(out_dir, 'p3_per_record.csv'), 'w', newline='', encoding='utf-8') as fh:
        w = csv.writer(fh)
        w.writerow(['отведения', 'пара', 'вариант', 'запись', 'годных циклов', 'в расчёте', '|r| ICA', '|r| регрессии',
                    'сходство ICA и регрессии', '|r| ритмической компоненты', 'причина исключения'])
        for (lead, pair, variant), res in results.items():
            for r in res['rows']:
                w.writerow([lead, pair, variant, r['record'], r['cycles'], 'да' if r['used'] else 'нет'] +
                           ([round(r[k], 4) for k in ('r_ica', 'r_ols', 'r_sim', 'r_rate')] if r['used'] else ['', '', '', '']) +
                           [r['reason']])
    summary['results'] = {f'{l}|{p}|{v}': res['summary'] for (l, p, v), res in results.items()}
    json.dump(_clean(summary), open(os.path.join(out_dir, 'p3_summary.json'), 'w', encoding='utf-8'),
              ensure_ascii=False, indent=1)
    print(f'\nРезультаты сохранены в папку {out_dir}')
    return dict(summary=summary, results=results)


def _in_notebook():
    """True, если код выполняется в Jupyter / Google Colab."""
    try:
        get_ipython  # noqa: F821
        return True
    except NameError:
        return False


if __name__ == '__main__' and not _in_notebook():
    ap = argparse.ArgumentParser(description='П.3: ICA на парах рядов RR, PP, PR')
    ap.add_argument('data_dir', help='папка с записями 00*_ecg.edf')
    ap.add_argument('--out', default='results_p3', help='папка для результатов')
    ap.add_argument('--no-synthetic', action='store_true', help='пропустить синтетическую проверку')
    a = ap.parse_args()
    run_p3(a.data_dir, a.out, with_synthetic=not a.no_synthetic)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 13.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyedflib: filename=pyedflib-0.1.42-cp313-cp313-linux_x86_64.whl size=2852054 sha256=395a0c73aa3ce2f683baebc04bf76c366db139fd7ef8c6dd546cb93e5ab3dbd6
  Stored in directory: /root/.cache/pip/wheels/ac/16/3e/5299c8a6b023bfd3d7204240ca5de940a49bede1edb30015ab
Successfully built pyedflib


In [2]:
run_p3('/content/data/')

Записей: 20. Лучшее отведение по записям: 0011 — II, 0012 — II, 0013 — II, 0014 — II, 0015 — II, 0016 — I, 0017 — III, 0018 — II, 0019 — II, 0020 — III, 0021 — II, 0022 — II, 0023 — II, 0024 — II, 0025 — III, 0026 — II, 0027 — II, 0028 — I, 0029 — II, 0030 — II
Строю ряды RR, PP, PR...
Вариант без уточнения положений R и P (отведение II)...
  без уточнения, PP+PR: ICA 0.613, регрессия 0.696, ICA лучше 5 из 20, p = 0.0215
  без уточнения, RR+PR: ICA 0.620, регрессия 0.700, ICA лучше 3 из 20, p = 0.0037
Тождество RR(k) = PP(k) + PR(k+1) − PR(k): максимальное отклонение 2.27e-13 мс по 13721 циклам

сочетание                                   записей   ICA    регр.  разрыв  ICA лучше      p
II   PP+PR  целиком                         20  0.541  0.595  +0.013    7 из 20  0.0826
II   PP+PR  целиком, порог 250              20  0.541  0.595  +0.013    7 из 20  0.0826
II   PP+PR  без выбросов                    20  0.458  0.493  +0.031    6 из 20  0.0897
II   PP+PR  покой                       

{'summary': {'identity': {'max_abs_ms': 2.2737367544323206e-13,
   'cycles': 13721},
  'unrefined_ii': {'PP+PR': {'n': 20,
    'med_ica': 0.6132526223931937,
    'med_ols': 0.696474995584638,
    'gap': 0.023658782089991115,
    'better': 5,
    'p': 0.021484375,
    'med_sim': 0.9444546668594394,
    'med_rate': 0.9453346883863795},
   'RR+PR': {'n': 20,
    'med_ica': 0.6197085286621582,
    'med_ols': 0.6996942817683616,
    'gap': 0.033917984994818684,
    'better': 3,
    'p': 0.00365447998046875,
    'med_sim': 0.9326981137298724,
    'med_rate': 0.9313925893770496}},
  'robustness': {'II|PP+PR|целиком|старты': {'starts': 20,
    'gap_min': 0.005959107463829716,
    'gap_max': 0.02751079350020058,
    'p_min': 0.03623390197753906,
    'p_max': 0.15364646911621094,
    'worse_and_significant': 3,
    'worse': 20},
   'лучшее|PP+PR|целиком|старты': {'starts': 20,
    'gap_min': 0.005959107463829716,
    'gap_max': 0.018588406519172068,
    'p_min': 0.05316925048828125,
    'p_max':